# Modelagem: curva de demanda final e previsões em kg

Este notebook consome a especificação definida no EDA: **OLS em `ln(volume)`**, quatro interceptos de nível — Segunda, Terça+Quarta+Quinta, Sexta e Fim de Semana —, **elasticidade-preço compartilhada** e correção global de **smearing** para retornar a previsão em quilogramas.

A implementação está centralizada em `src/modeling/demand_curve.py`. Nenhuma observação é removida; Huber permanece como checagem de robustez documentada no EDA, não como o modelo operacional.

In [1]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.modeling.demand_curve import (
    CLUSTERS,
    assign_cluster,
    fit_demand_curve,
    is_within_price_support,
    predict_volume,
)



Fontconfig error: No writable cache directories
	/opt/homebrew/var/cache/fontconfig
	/Users/ngodeny/.cache/fontconfig
	/Users/ngodeny/.fontconfig

Fontconfig error: No writable cache directories
	/opt/homebrew/var/cache/fontconfig
	/Users/ngodeny/.cache/fontconfig
	/Users/ngodeny/.fontconfig

Fontconfig error: No writable cache directories
	/opt/homebrew/var/cache/fontconfig
	/Users/ngodeny/.cache/fontconfig
	/Users/ngodeny/.fontconfig

Fontconfig error: No writable cache directories
	/opt/homebrew/var/cache/fontconfig
	/Users/ngodeny/.cache/fontconfig
	/Users/ngodeny/.fontconfig



Fontconfig error: No writable cache directories
	/opt/homebrew/var/cache/fontconfig
	/Users/ngodeny/.cache/fontconfig
	/Users/ngodeny/.fontconfig


Matplotlib is building the font cache; this may take a moment.


2026-09-19 17:41:48.305 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: /Users/ngodeny/Documents/mestrado/202602-resolucao_problemas


## Dados disponíveis

As 76 primeiras observações são usadas para ajustar a curva. Os 10 dias finais continuam separados do ajuste, mas já foram consultados durante a exploração de alternativas; por isso, as métricas abaixo são **descritivas e exploratórias**, não uma estimativa final independente de desempenho fora da amostra.

In [2]:
treino = pd.read_csv(ROOT / "data/interim/.treino.csv", parse_dates=["Data"])
teste = pd.read_csv(ROOT / "data/interim/.teste.csv", parse_dates=["Data"])

print(f"Treino: {len(treino)} dias, de {treino['Data'].min():%d/%m/%Y} a {treino['Data'].max():%d/%m/%Y}")
print(f"Referência exploratória: {len(teste)} dias, de {teste['Data'].min():%d/%m/%Y} a {teste['Data'].max():%d/%m/%Y}")
display(treino[["Data", "Dia da Semana", "Preço", "Volume Realizado (kg)"]].head())

Treino: 76 dias, de 01/08/2025 a 31/10/2025
Referência exploratória: 10 dias, de 03/11/2025 a 14/11/2025


,Data,Dia da Semana,Preço,Volume Realizado (kg)
0,2025-08-01,Sexta,1267.949227,5.625635
1,2025-08-04,Segunda,1279.335771,31.390461
2,2025-08-05,Terça,1258.408986,22.184888
3,2025-08-06,Quarta,1241.133798,11.030794
4,2025-08-07,Quinta,1253.118993,15.160805


## Ajuste do modelo final

A função `fit_demand_curve` executa OLS no log do volume, calcula o fator de smearing a partir dos resíduos do próprio treino e guarda o suporte de preço observado em cada cluster. A matriz de desenho é fixa, com TerQuaQui como referência, para que treino e previsão usem a mesma especificação.

In [3]:
modelo_demanda = fit_demand_curve(treino)

coeficientes = modelo_demanda.params.rename("Coeficiente").to_frame()
coeficientes.index.name = "Termo"
display(coeficientes.round(6))

print(
    "ln(volume) = "
    f"{modelo_demanda.params['const']:.6f} "
    f"{modelo_demanda.params['ln_preco']:+.6f}·ln(preço) "
    f"{modelo_demanda.params['Segunda']:+.6f}·Segunda "
    f"{modelo_demanda.params['Sexta']:+.6f}·Sexta "
    f"{modelo_demanda.params['FimDeSemana']:+.6f}·FDS"
)
print(f"Fator global de smearing: {modelo_demanda.smearing_factor:.6f}")

suporte_preco = pd.DataFrame(
    [(cluster, minimo, maximo) for cluster, (minimo, maximo) in modelo_demanda.price_support.items()],
    columns=["Cluster", "Preço mínimo observado", "Preço máximo observado"],
).set_index("Cluster").loc[list(CLUSTERS)]
display(suporte_preco.round(2))

,Coeficiente
Termo,
const,92.174900
ln_preco,-12.458103
Segunda,0.338230
Sexta,-1.255048
FimDeSemana,-2.883045


ln(volume) = 92.174900 -12.458103·ln(preço) +0.338230·Segunda -1.255048·Sexta -2.883045·FDS
Fator global de smearing: 1.117873


,Preço mínimo observado,Preço máximo observado
Cluster,,
Segunda,1220.47,1352.42
TerQuaQui,1201.63,1344.63
Sexta,1208.37,1365.49
FimDeSemana,1188.86,1354.19


## Referência exploratória de previsão

As previsões são feitas em kg e aplicam smearing. A função bloqueia extrapolações por padrão; neste recorte, todos os preços dos 10 dias estão dentro da faixa observada para seu cluster. RMSE, WMAPE, MAPE por incidência e viés são apresentados para caracterizar o resultado já observado, sem alegar validação externa limpa.

In [4]:
avaliacao = teste.copy()
avaliacao["Cluster"] = avaliacao["Dia da Semana"].map(assign_cluster)
avaliacao["No suporte"] = avaliacao.apply(
    lambda linha: is_within_price_support(modelo_demanda, linha["Preço"], linha["Cluster"]), axis=1
)
assert avaliacao["No suporte"].all(), "Há preços de avaliação fora do suporte do respectivo cluster."
avaliacao["Volume previsto (kg)"] = avaliacao.apply(
    lambda linha: predict_volume(modelo_demanda, linha["Preço"], linha["Cluster"]), axis=1
)
real = avaliacao["Volume Realizado (kg)"].to_numpy()
previsto = avaliacao["Volume previsto (kg)"].to_numpy()
erro = real - previsto
metricas = pd.DataFrame([{
    "RMSE (kg)": np.sqrt(np.mean(erro ** 2)),
    "WMAPE volume (%)": np.abs(erro).sum() / real.sum() * 100,
    "MAPE incidência (%)": (np.abs(erro) / real).mean() * 100,
    "Viés agregado (%)": (previsto.sum() - real.sum()) / real.sum() * 100,
}])
display(metricas.round(2))
avaliacao["Erro absoluto (%)"] = np.abs(erro) / real * 100
display(avaliacao[["Data", "Dia da Semana", "Cluster", "Preço", "Volume Realizado (kg)", "Volume previsto (kg)", "Erro absoluto (%)"]].round(2))

,RMSE (kg),WMAPE volume (%),MAPE incidência (%),Viés agregado (%)
0,8.09,34.78,58.07,-17.22


/var/folders/6g/yn7cwj_52f72rdbklyyb3hsh0000gn/T/ipykernel_96959/3599295518.py:21: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(avaliacao[["Data", "Dia da Semana", "Cluster", "Preço", "Volume Realizado (kg)", "Volume previsto (kg)", "Erro absoluto (%)"]].round(2))


,Data,Dia da Semana,Cluster,Preço,Volume Realizado (kg),Volume previsto (kg),Erro absoluto (%)
0,2025-11-03,Segunda,Segunda,1343.80,29.91,17.92,40.09
1,2025-11-04,Terça,TerQuaQui,1294.32,9.47,20.39,115.33
2,2025-11-05,Quarta,TerQuaQui,1311.53,16.43,17.29,5.22
3,2025-11-06,Quinta,TerQuaQui,1334.94,17.26,13.87,19.61
4,2025-11-07,Sexta,Sexta,1327.98,1.18,4.22,258.86
5,2025-11-10,Segunda,Segunda,1334.80,18.19,19.48,7.12
6,2025-11-11,Terça,TerQuaQui,1302.99,27.57,18.76,31.97
7,2025-11-12,Quarta,TerQuaQui,1311.07,21.37,17.37,18.72
8,2025-11-13,Quinta,TerQuaQui,1318.27,32.47,16.22,50.04
9,2025-11-14,Sexta,Sexta,1282.92,9.79,6.49,33.75


## Curvas finais do modelo operacional

Cada painel mostra observações de treino e a curva OLS já corrigida por smearing. A curva é calculada somente entre o menor e o maior preço observados no próprio cluster. O bloqueio de extrapolação da API mantém essa mesma condição nas previsões futuras.

In [5]:
cores = {
    "Segunda": "#1f77b4",
    "TerQuaQui": "#2ca02c",
    "Sexta": "#ff7f0e",
    "FimDeSemana": "#9467bd",
}
dias_representativos = {
    "Segunda": "Segunda",
    "TerQuaQui": "Terça",
    "Sexta": "Sexta",
    "FimDeSemana": "Sábado",
}
treino_plot = treino.copy()
treino_plot["Cluster"] = treino_plot["Dia da Semana"].map(assign_cluster)

fig, eixos = plt.subplots(2, 2, figsize=(12, 8))
for ax, cluster in zip(eixos.flat, CLUSTERS):
    observados = treino_plot[treino_plot["Cluster"] == cluster]
    minimo, maximo = modelo_demanda.price_support[cluster]
    precos = np.linspace(minimo, maximo, 200)
    volumes = [predict_volume(modelo_demanda, preco, cluster) for preco in precos]
    ax.scatter(observados["Preço"], observados["Volume Realizado (kg)"], color=cores[cluster], alpha=0.7, s=45, label="Observado")
    ax.plot(precos, volumes, color=cores[cluster], lw=2.8, label="OLS + smearing")
    ax.set(title=cluster, xlabel="Preço realizado (R$/kg)", ylabel="Volume (kg)")
    ax.grid(alpha=0.25)
    ax.legend()
fig.suptitle(f"Curvas finais: OLS com elasticidade compartilhada (β={modelo_demanda.params['ln_preco']:.2f})", y=1.01)
fig.tight_layout()
plt.show()

/var/folders/6g/yn7cwj_52f72rdbklyyb3hsh0000gn/T/ipykernel_96959/2326686596.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Limites e ponte para otimização

- A curva representa a melhor especificação escolhida com a base disponível, mas a elasticidade mantém incerteza material documentada por LOO, bootstrap e Huber no EDA.
- FDS mantém intercepto próprio; não possui observação no período separado e deve ser acompanhado quando novos fins de semana estiverem disponíveis.
- Para a otimização, o domínio de preços deve ser limitado ao suporte observado de cada cluster. A função `predict_volume` exige isso por padrão.
- Caso novos dados sejam incorporados, a curva e o fator de smearing devem ser reajustados somente com o treino disponível naquele momento.